# Sliding Dot product, (Circular) Convolution, and Overlap-Add!

In [2]:
import numpy as np
import scipy
import time

## Introduction

One way to compute the sliding-dot-product (sdp) between a query Q and a time series T is
FFT-based convolution. But first, let's start with a simple example to understand how these two concepts are related.


Q = [A, B]
<br>
T = [1, 2, 3, 4]


Their sdp is:
<br>    
sdp(Q, T) = [1*A + 2*B, 2*A + 3*B, 3*A + 4*B]


## How is it related to convolution?

There are different types of convolutions. The one we are interested in is circular convolution. The following formula computes the circular convolution between two arrays `X` and `Y`, each with length `N`

$$CONV_{X,Y}[i] = \sum_{j=0}^{N-1} X[j] \cdot Y[(i - j) \mod N]$$

where `N` is the length of X and Y. 

Let's set $X$  to `T`, where `len(T)==N`. Let's pad `Q[::-1]` with 0s till its length becomes `N`, and asign it to $Y$. Then, we will see that part of their convolution is `sdp(Q, T)`. Let's see:

```
# circular convolution between [1, 2, 3, 4] and [B A 0 0]
index 0: 1B + 4A
index 1: 1A + 2B
index 2: 2A + 3B
index 3: 3A + 4B
```

We can see that the slice `[M-1: N]`, where `M=len(Q)=2` and `N=len(T)=4`, represents `sdp`.

## How is it related to FFT-based convolution?

It can be shown that the circular convolution above can be computed with help of FFT as follows:

$$CONV_{X,Y} = IFFT(FFT(X) * FFT(Y))$$

Let's check this... 

In [3]:
def circular_convolution_direct(X, Y):
    N = len(X)
    out = np.zeros(N, dtype=np.float64)
    for i in range(N):
        for j in range(N):
            out[i] += X[j] * Y[(i-j) % N]

    return out


def circular_convolution_fft(X, Y):
    return scipy.fft.irfft(scipy.fft.rfft(X) * scipy.fft.rfft(Y))

In [4]:
X = np.random.rand(10)
Y = np.random.rand(10)

ref = circular_convolution_direct(X, Y)
comp = circular_convolution_fft(X, Y)
np.testing.assert_allclose(ref, comp)

The assertion is passing. This confirms that the fft-based method for convolving the two arrays gives the same result as the circular convolution. To the best of my knowledge,there is no function in numpy or scipy to give us the circular convolution. `scipy.signal.fftconvolve` computes the linear convolution which result in differen output. However, the good news is that the slice `M-1 : N` still gives the sliding dot product. 

In [7]:
def circular_fftconvolution_sdf(Q, T):
    m = len(Q)
    n = len(T)

    Q_flip_padded = np.empty(n, dtype=np.float64)
    Q_flip_padded[:m] = Q[::-1]
    Q_flip_padded[m:] = 0
    QT_concolve = circular_convolution_fft(T, Q_flip_padded)
    
    return QT_concolve[m-1:n]


def linear_fftconvolve_sdp(Q, T):
    # scipy performs linear convolution
    return scipy.signal.fftconvolve(T, Q[::-1], mode='valid')  # valid gives the slice `[M - 1: N]`


T = np.random.rand(10)
Q = np.random.rand(3)

ref = circular_fftconvolution_sdf(Q, T)
comp = linear_fftconvolve_sdp(Q, T)

np.testing.assert_allclose(ref, comp)

**A couple of notes:**

(1) For two arrays X and Y with length m and n, their linear convolution is computed by padding 0s at the end of each array till its legnth becomes `m + n - 1`. We can check this as follows:

In [8]:
def linear_convolution(X, Y):
    return scipy.signal.fftconvolve(X, Y, mode='full')

def linear_convolution_mirror(X, Y):
    n = len(X) + len(Y) - 1
    
    X_padded = np.empty(n, dtype=np.float64)
    X_padded[:len(X)] = X
    X_padded[len(X):] = 0

    Y_padded = np.empty(n, dtype=np.float64)
    Y_padded[:len(Y)] = Y
    Y_padded[len(Y):] = 0

    return circular_convolution_fft(X_padded, Y_padded)


X = np.random.rand(1024)
Y = np.random.rand(128 + 1)

ref = linear_convolution(X, Y)
comp = linear_convolution_mirror(X, Y)
np.testing.assert_allclose(ref, comp)

I did a a little bit cheating here.... the assetion above fails if we decide to go with `Y = np.random.rand(128 + 1)`. This is because scipy uses next_fast_len which was not considered in our `linear_convolution_mirror` function...but I think the main message was delivered... the linear convolution is the same as the circular one when arrays are padded till their lengths become `len(X) + len(Y) - 1`. 

(2) Because linear convolution applies (r)fft on longer arrays, it should be slower than circular. Let's check that out!

In [9]:
#####
T = np.random.rand(2 ** 20)
Q = np.random.rand(2 ** 19 + 1)

timeout = 60

#########
total_time = 0
n_iter = 0
ref = linear_fftconvolve_sdp(Q, T)
while total_time < timeout:
    start = time.time()
    linear_fftconvolve_sdp(Q, T)
    total_time += time.time() - start
    n_iter += 1

ref_timing = total_time / n_iter
print('sdp timing using linear convolution --> ', ref_timing)
##########
total_time = 0
n_iter = 0
comp = circular_fftconvolution_sdf(Q, T)
while total_time < timeout:
    start = time.time()
    circular_fftconvolution_sdf(Q, T)
    total_time += time.time() - start
    n_iter += 1

comp_timing = total_time / n_iter
print('sdp timing using circular convolution on `max(len(Q), len(T))` --> ', comp_timing)

performance_ratio = ref_timing / comp_timing
print(f'Circular convolution is {performance_ratio} times faster than linear convolution for sdp.')

# assert outputs
np.testing.assert_allclose(ref, comp)

sdp timing using linear convolution -->  0.04477471882153766
sdp timing using circular convolution on `max(len(Q), len(T))` -->  0.0248597981817187
Circular convolution is 1.8010893931739123 times faster than linear convolution for sdp.


We can see that circular convolution is 80% faster in this case. Note that the length of Q was purposefully chosen to be large so that its impact on `len(T) + len(Q) - 1` (the length of arrays for linear convolution) becomes clearer.

## Overlap-add Convolution

Let's consider the folllwing example:

```
Q = [A, B]
T = [1, 2, 3, 4, 5, 6]
```

And their sdp is: `sdp(Q, T) = [1A + 2B, 2A + 3B, 3A + 4B, 4A + 5B, 5A + 6B]`. 

What if I break down T into two non-overlapping chunks `T1=[1, 2, 3]` and `T2=[4, 5, 6]`? Let's compute the sdp for each:

```
sdp(Q, T1) = [1A + 2B, 2A + 3B]
sdp(Q, T2) = [4A + 5B, 5A + 6B]
```

If I put these elements together, I can see that the element `3A + 4B` is missing. This is because chunking the time series destroyed one of the subsequences. Is there a way to not lose that element? We basically need to find a way to construct that element. `3A + 4B` can be written as: `(3A + 0B) + (0A + 4B)`. If we look closely, we can see:

* The first component, `3A + 0B` is the dot product of `[3, 0]` and `[A, B]` 
* The second component, `0A + 4B` is the dot product of `[0, 4]` and `[A, B]`

So, what if we chunk and pad it with 0?

```
T1' = [1, 2, 3, 0] --> sdp(Q, T1) = [1A + 2B, 2A + 3B, 3A + 0B]
T2' = [0, 4, 5, 6] --> sdp(Q, T1) = [0A + 4B, 4A + 5B, 5A + 6B]
```

And now I can add the last element of `sdp(Q, T1)` to the first element of `sdp(Q, T2)` to get that missing element`3A + 4B`!

Great! Now, we will show that our approach still works if we add `0` at the end of `T2`, and use circular convolution!

```
# chunk T
T1' = [1, 2, 3, 0]  
T2' = [4, 5, 6, 0] 

# flip Q and pad it with zero
Q' =  [B, A, 0, 0]    
```

**and their corresponding circular convolutions are:**

* circular_convolution(Q', T1'): [1B, 1A + 2B, 2A + 3B, **3A + 0B**]
* circular_convolution(Q', T1'): [**0A + 4B**, 4A + 5B, 5A+6B, 6A + 0B]



Note that the last element of each convolution is meaningless unless it is added to the first element of the convolution of next chunk. Therefore, we can:

(1) Slice the output upto a point where zero-padding started
<br>
(2) add last element of convolution of a chunk to the first element of convolution of next chunk
<br>
(3) Put the elements together, flatten the result, and get the slice `[M - 1 : N]`

And this will give us: <br>
`[1A + 2B, 2A + 3B, (3A + 0B) + (0A + 4B), 4A + 5B, 5A + 6B]`

In our example, `m=2`, and `m-1=1` zero was used for padding. In general, the following approach can be taken for overlap-add:

* Choose a `block_size`: this is the size of each chunk AFTER zero padding.
* Based on the `block_sise`, Find the `chunk_size = block_size - (m - 1)`: this is the chunk size when chunking `T`
* Calculate circular convolution between each chunk and `Q[::-1]`, and use overlap-add logic to caculate the sdp for subsequneces that are destoried because of the chunking.